# Week 3 — Introduction to scikit-learn I
## From a data table to a justified prediction

**Guiding question:** *Can we predict an order's packing time before packing begins, and how should we choose a model?*

This notebook is preparation material for independent study. Work through the complete examples in order and pause at the self-checks before reading the explanations. The main walkthrough runs without filling in code. Optional investigations appear at the end.

We use the same simulated orders in `data/orders.csv` that you previewed in Week 2. This week we deliberately restrict the model inputs to total item count and fragile item count. The observed packing times come from the full, more complicated process; selecting fewer columns does not simplify how those times were generated.

The data illustrate modelling decisions. They are not records from a real warehouse.

### Learning goals

By the end of this notebook, you will be able to:

- distinguish describing data, simulating a process, and learning a prediction rule;
- define the observation, prediction time, features, and target;
- assemble a suitable modelling table and recognise data leakage;
- separate training, validation, and test observations;
- create, fit, and use scikit-learn model objects;
- compare a median baseline with one- and two-feature regressions using MAE;
- interpret predictions, signed errors, and the size of an improvement;
- refit the selected approach and assess it on held-out test data; and
- predict new orders and explain the limits of extrapolation.

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead><tr><th colspan="2">Contents</th></tr></thead>
<tbody>
<tr><td>1</td><td><a href="#section-from-description-to-prediction">From description to prediction</a></td></tr>
<tr><td>2</td><td><a href="#section-assemble-the-dataset">Assemble the dataset</a></td></tr>
<tr><td>3</td><td><a href="#section-separate-learning-selection-and-testing">Separate learning, selection, and testing</a></td></tr>
<tr><td>4</td><td><a href="#section-explore-training-orders">Explore training orders</a></td></tr>
<tr><td>5</td><td><a href="#section-measure-prediction-error">Measure prediction error</a></td></tr>
<tr><td>6</td><td><a href="#section-fit-a-median-baseline">Fit a median baseline</a></td></tr>
<tr><td>7</td><td><a href="#section-fit-one-feature-regression">Fit a one-feature regression</a></td></tr>
<tr><td>8</td><td><a href="#section-add-a-second-feature">Add a second feature</a></td></tr>
<tr><td>9</td><td><a href="#section-compare-predictions-and-errors">Compare predictions and errors</a></td></tr>
<tr><td>10</td><td><a href="#section-select-refit-and-test">Select, refit, and test</a></td></tr>
<tr><td>11</td><td><a href="#section-predict-new-orders">Predict new orders</a></td></tr>
<tr><td>12</td><td><a href="#section-recap-and-outlook">Recap and outlook</a></td></tr>
<tr><td>★</td><td><a href="#section-optional-your-turn">Optional: Your turn</a></td></tr>
</tbody></table>

### How to work with this notebook

Execute the code cells from top to bottom with **Shift + Enter**. Each section creates objects that later cells use. If a result seems inconsistent, restart the kernel and run all cells.

The code comments explain individual steps. At each **Self-check**, stop and make a prediction or interpretation before continuing. The purpose is not only to produce an output, but to understand what question the output answers.

<h2 id="section-from-description-to-prediction">1. From description to prediction</h2>

Week 2 answered descriptive questions such as “What is the average sales value?” It also generated data using a formula and chosen distribution settings. In a **simulation**, we specify a process and generate possible observations. In **prediction modelling**, we estimate a rule from examples and apply it to new observations.

Suppose a completed order contained five items and took eight minutes to pack. Another order contained fifteen items and took twenty-five minutes. These pairs of inputs and observed outcomes provide examples from which a model can learn.

This is **supervised learning**: the training examples include the outcomes we want to predict. Here the outcome is a numerical duration, so the task is **regression**. The model does not receive the actual duration when predicting a new order.

For example, a simple rule might predict two extra minutes for each additional item. The training procedure estimates such numerical quantities from the examples; they are called the model's **parameters**. These differ from the function parameters introduced in Week 1: here the word refers to learned numbers stored in the fitted model.

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead>
<tr><th>Term</th><th>Meaning in our example</th></tr>
</thead>
<tbody>
<tr><td>Observation</td><td>One order</td></tr>
<tr><td>Feature</td><td>An input available before packing, such as item count</td></tr>
<tr><td>Target</td><td>The observed packing time we want to predict</td></tr>
<tr><td>Prediction</td><td>The model's estimated packing time for an order</td></tr>
<tr><td>Fitting / training</td><td>Estimating the model's parameters from examples</td></tr>
</tbody>
</table>

A model can find a useful predictive association without identifying a causal effect. A relationship learned from these orders does not automatically tell us what would happen if we intervened in a real warehouse.

### Introducing scikit-learn

**scikit-learn** supplies model classes, data-splitting tools, and evaluation functions. Its import name is `sklearn`. The familiar objects and methods now represent statistical models.

The imports below select particular tools from the package's modules. For example, `from sklearn.linear_model import LinearRegression` makes the class name `LinearRegression` available directly.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

<h2 id="section-assemble-the-dataset">2. Assemble the dataset</h2>

The notebook and the `data` folder are both in the repository root. As in Week 2, read the CSV into a DataFrame and inspect its structure.

Our question concerns predictions **before packing begins**. Item counts are known then. The actual packing time is measured after packing. It is useful as a historical training label, but not as an input for a new prediction.

In [ ]:
data_path = Path("data/orders.csv")
all_orders = pd.read_csv(data_path)

print("Rows and columns:", all_orders.shape)
print("Column names:", all_orders.columns.tolist())
all_orders.head()

### Choose information for a specific question

The CSV contains seven possible input features. We begin with two; Week 4 considers all seven.

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead>
<tr><th>Column</th><th>Role in this walkthrough</th></tr>
</thead>
<tbody>
<tr><td><code>order_id</code></td><td>Identifier retained for checking rows, not used as an input</td></tr>
<tr><td><code>item_count</code></td><td>First feature: total items, including fragile items</td></tr>
<tr><td><code>fragile_count</code></td><td>Second feature: fragile items within that total</td></tr>
<tr><td><code>packing_time_min</code></td><td>Target: actual duration in minutes</td></tr>
<tr><td><code>storage_zones</code>,<br><code>gift_wrap</code>,<br><code>total_weight_kg</code>,<br><code>order_value_eur</code>,<br><code>package_side_cm</code></td><td>Available information not supplied to our three candidates</td></tr>
</tbody>
</table>

An order with five total items and two fragile items has three non-fragile items, not seven items in total. This matters when interpreting coefficients.

The table selection below keeps **all orders** but only four columns. We create a copy so later changes to this working table do not alter `all_orders`. The feature lists contain the input column names in their intended order.

In [ ]:
features_one = ["item_count"]
features_two = ["item_count", "fragile_count"]
target = "packing_time_min"

columns_to_keep = ["order_id"] + features_two + [target]
orders = all_orders[columns_to_keep].copy()

orders.head()

### Check the meaning and quality of rows

The inspection commands come from Week 2. We also check repeated identifiers: `.duplicated()` marks repeated values after their first occurrence, and summing the resulting Booleans counts them.

Our rules describe valid orders: counts must be whole numbers, total items positive, fragile items between zero and total items, and durations positive. A row with a missing value needs investigation; missing does not mean zero.

These checks examine structure and validity before modelling. They do not select features by comparing their predictive performance. We reserve exploration of relationships for the training data.

In [ ]:
print(orders.dtypes)
print("\nMissing values per column:")
print(orders.isna().sum())
print("\nRepeated order IDs:", orders["order_id"].duplicated().sum())

# Stop visibly if the supplied table violates an expected condition.
assert orders["order_id"].is_unique
assert not orders.isna().any().any()
assert (orders["item_count"] > 0).all()
assert (orders["fragile_count"] >= 0).all()
assert (orders["fragile_count"] <= orders["item_count"]).all()
assert (orders["item_count"] % 1 == 0).all()
assert (orders["fragile_count"] % 1 == 0).all()
assert (orders[target] > 0).all()

print("All structural and validity checks passed.")

The supplied table passes these checks. The two `.any()` calls first ask whether each column has a missing value and then whether any column does. `not` reverses that answer for the assertion. The remainder check `% 1 == 0` verifies whole-number counts.

A repeated order ID might mean that the same observation was imported twice. Identical feature values alone do not prove duplication: two different orders can contain the same numbers of items.

In a real project, investigate failed checks before deciding whether to correct, exclude, or retain rows. Unusual but valid long orders should not be removed merely because a model predicts them poorly. Units also matter: 120 seconds and 120 minutes are very different targets.

### Leakage: information that gives an unfair advantage

**Data leakage** occurs when model fitting or model selection uses information that would not legitimately be available for the intended prediction or evaluation.

Giving `packing_time_min` to a model that predicts packing time would reveal the answer. Letting copies of the same order enter both training and evaluation would also make evaluation misleading.

The sequential identifier is numeric, but it is not a meaningful description of an order in this simulation. Numeric storage alone is not a reason to include a column.

**Self-check:** Why may the target be present in the historical table while still being excluded from the feature list?

The historical target tells the training procedure what happened. For a new order we only know the inputs. We therefore separate input columns from the target so that fitting can use both, while prediction uses only the inputs.

<h2 id="section-separate-learning-selection-and-testing">3. Separate learning, selection, and testing</h2>

A model can fit the examples it has seen without predicting new cases equally well. **Generalisation** means applying the learned rule usefully to unseen observations. We need data that were not used to fit a model to assess this.

We also want to choose between candidates. If we choose a candidate because it performs well on certain observations, those observations have influenced our choice. They are no longer an independent final check.

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead>
<tr><th>Part</th><th>Purpose</th><th style="text-align: right;">Share of all orders</th></tr>
</thead>
<tbody>
<tr><td>Training</td><td>Learn each candidate's parameters</td><td style="text-align: right;">60%</td></tr>
<tr><td>Validation</td><td>Compare candidates and make the choice</td><td style="text-align: right;">20%</td></tr>
<tr><td>Test</td><td>Evaluate the fixed final choice</td><td style="text-align: right;">20%</td></tr>
</tbody>
</table>

We call training and validation together the **development data**. Test data remain separate until selection is complete. These percentages are a practical, common example, not a universal prescription.

### Make the split in two stages

First reserve 20% as test data. Then split the remaining 80%: one quarter of it becomes validation, because `0.25 × 0.80 = 0.20`. The other three quarters become training: `0.75 × 0.80 = 0.60`.

`train_test_split` returns two subsets. The argument `test_size` names the size of its second output; that output can serve as our validation subset in the second call. The function does not decide the scientific role of the data—we do.

Splitting whole rows keeps each order's features and target together. `random_state=42` makes the random allocation reproducible, much like a seed in Week 2. It is not a performance setting to optimise.

**Self-check:** For 2,000 orders, how many rows should each of the three parts contain?

In [ ]:
development, test = train_test_split(
    orders,
    test_size=0.20,
    random_state=42,
)

train, validation = train_test_split(
    development,
    test_size=0.25,
    random_state=42,
)

split_sizes = pd.DataFrame({
    "part": ["Training", "Validation", "Test"],
    "orders": [len(train), len(validation), len(test)],
})
split_sizes["share"] = split_sizes["orders"] / len(orders)
split_sizes

The counts are 1,200 training, 400 validation, and 400 test orders. Using `test_size=0.20` in both calls would instead give 64% training and 16% validation, because the second percentage applies to the remaining rows.

The random split suits the independently simulated orders in this example. If predicting future orders from historical records, a past/future split may be more appropriate. If related observations share a customer or another unit, keeping that unit in one part may be necessary. A fixed seed reproduces a split; it does not make an unsuitable split appropriate.

### A new rule for familiar preprocessing

In Week 2, replacing a missing price with a median illustrated a descriptive data-cleaning decision. In a predictive workflow, a median used to replace missing inputs is a quantity **learned from data**.

Calculate it on training rows only, then apply that same value to training and validation rows. Otherwise information from validation has entered the training procedure. The same principle applies to means and standard deviations used for scaling.

For example, if training prices are 10, 20, and 30, their median is 20. A missing validation price would receive 20, not a value recalculated from validation prices. Our selected columns are complete, so no replacement is needed here.

**Self-check:** Can we compare prediction errors on validation orders to choose a candidate and still call validation an independent final test?

No. Validation has guided the choice. That is why we reserve a separate test set. In Week 4, pipelines keep learned preprocessing inside each training split.

<h2 id="section-explore-training-orders">4. Explore training orders</h2>

Use the training rows to examine typical values and relationships. The target is already recorded in minutes, so our task is to select and understand it rather than reconstruct it from timestamps.

A summary shows the ranges and spread. A scatter plot then asks whether larger orders tend to take longer and whether fragile item count adds a visible distinction.

In [ ]:
train[features_two + [target]].describe().round(2)

### Add a third variable through colour

The scatter plot follows Week 2's `Figure` and `Axes` style. One point represents one training order.

- Horizontal position: total items.
- Vertical position: actual packing time.
- Colour: fragile items.

`c=` supplies one numeric colour value per point. `cmap="viridis"` maps values to a colour gradient. The colour bar explains the mapping. Transparency (`alpha`) helps with overlapping points, although it cannot reveal every hidden point.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")

points = ax.scatter(
    train["item_count"],
    train[target],
    c=train["fragile_count"],
    cmap="viridis",
    alpha=0.55,
    s=24,
)
fig.colorbar(points, ax=ax, label="Fragile items")
ax.set_xlabel("Total items (including fragile items)")
ax.set_ylabel("Actual packing time (minutes)")
ax.set_title("Training orders: inputs and observed duration")
ax.grid(alpha=0.2)
plt.show()

**Self-check:** If two orders have the same item count, must they take the same time? Would adding fragile count remove every difference?

Neither follows. Orders with similar counts still vary in weight, package size, wrapping requirements, and other characteristics. There is also random variation. Colour can suggest a pattern worth investigating, but does not quantify whether a second feature will improve predictions on new orders.

We therefore compare the candidates on the same validation orders. The plot alone does not establish which candidate is best.

<h2 id="section-measure-prediction-error">5. Measure prediction error</h2>

Before comparing candidates, we decide what “better” means. Our primary criterion is **mean absolute error (MAE)** on validation orders: smaller is better.

For one order, define the signed error as **prediction minus actual time**. Positive means overprediction; negative means underprediction. Taking the absolute value removes the sign. Averaging absolute errors gives MAE, in minutes.

This criterion assigns the same cost to an extra minute of error in either direction. That is a simplifying choice for this example. In a real planning decision, underestimating a delay and overestimating staff requirements may have different consequences.

**Self-check:** Actual times are 5, 8, and 11 minutes; predictions are 6, 6, and 14. Predict the signed errors, the absolute errors, and the MAE.

In [ ]:
example_actual = np.array([5, 8, 11])
example_predictions = np.array([6, 6, 14])

signed_errors = example_predictions - example_actual
absolute_errors = np.abs(signed_errors)

error_example = pd.DataFrame({
    "actual_minutes": example_actual,
    "predicted_minutes": example_predictions,
    "signed_error": signed_errors,
    "absolute_error": absolute_errors,
})
print(error_example)
print("\nMAE calculated with NumPy:", absolute_errors.mean())
print("MAE calculated with scikit-learn:",
      mean_absolute_error(example_actual, example_predictions))

The signed errors are 1, −2, and 3; their absolute values are 1, 2, and 3. The MAE is 2 minutes. Averaging signed errors instead would allow over- and underpredictions to cancel.

An MAE of 2 minutes means that the evaluated predictions differ from the actual durations by 2 minutes on average. It is **not** a maximum error, a percentage, or a prediction interval. Some individual errors can be much larger.

Our selection rule is fixed now: choose the smallest unrounded validation MAE among the three candidates. If values are exactly tied, prefer the simpler candidate in the order baseline, one feature, two features. We will also discuss the practical size of an improvement rather than treating every numerical difference as decisive.

<h2 id="section-fit-a-median-baseline">6. Fit a median baseline</h2>

A **baseline** is a simple reference against which to judge a more elaborate approach. Ours predicts the same value for every order: the **median training packing time**.

Week 2 introduced the median as the middle ordered value. It is useful here because a median minimises the sum of absolute distances to the training values. It gives us a meaningful constant predictor for MAE.

The baseline learns its constant from training durations only. Calculating the median from the entire dataset would use outcomes we are reserving for evaluation.

### Prepare the model inputs

We use the conventional name `X` for a feature table and `y` for the target sequence. The distinction between Series and DataFrame was introduced in Week 2; it now matters for the model interface.

Even one feature is passed as a **two-dimensional table**: `train[features_one]`, where `features_one` is a list. The target is a **one-dimensional Series**: `train[target]`, where `target` is a single column name. Corresponding rows must describe the same orders.

In [ ]:
X_train_one = train[features_one]
X_validation_one = validation[features_one]
y_train = train[target]
y_validation = validation[target]

print("Training feature table:", X_train_one.shape)
print("Training target sequence:", y_train.shape)
print("Validation feature table:", X_validation_one.shape)

### Create, fit, predict, evaluate

`DummyRegressor(strategy="median")` creates a model object. Its `.fit()` method learns the training median. Its `.predict()` method returns one value per supplied input row.

The method notation is familiar from Week 1's `Company` class and Week 2's package objects. What changes is the operation: the model stores quantities estimated from data.

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead>
<tr><th>Step</th><th>Information supplied</th><th>Result</th></tr>
</thead>
<tbody>
<tr><td>Create</td><td>Model settings</td><td>Unfitted object</td></tr>
<tr><td>Fit</td><td>Training features and targets</td><td>Object containing learned quantities</td></tr>
<tr><td>Predict</td><td>Features for the orders of interest</td><td>Predicted durations</td></tr>
<tr><td>Evaluate</td><td>Actual durations and predictions</td><td>Error measure</td></tr>
</tbody>
</table>

The baseline ignores feature values, but accepts a feature table so that its interface matches other models.

In [ ]:
baseline = DummyRegressor(strategy="median")

# Learn the constant using training outcomes only.
baseline.fit(X_train_one, y_train)

# Validation outcomes are not passed to predict().
prediction_baseline = baseline.predict(X_validation_one)
mae_baseline = mean_absolute_error(y_validation, prediction_baseline)

print(f"Training median: {y_train.median():.2f} minutes")
print("First five predictions:", prediction_baseline[:5].round(2))
print(f"Baseline validation MAE: {mae_baseline:.2f} minutes")

Every prediction is identical, even though the actual orders differ. This is intentional: a more elaborate model should demonstrate that the inputs help.

**Self-check:** Why does `.fit()` need historical outcomes while `.predict()` does not?

Fitting uses known outcomes to estimate a rule. Prediction applies the stored rule to inputs. Actual validation outcomes are needed afterwards to measure the errors, not to produce the predictions.

<h2 id="section-fit-one-feature-regression">7. Fit a one-feature regression</h2>

A one-feature linear regression predicts duration using a straight line:

**predicted minutes = intercept + item coefficient × total items**

The **intercept** is the line's predicted value at zero items. The **coefficient** is the change in predicted minutes for one additional item. These quantities are learned parameters.

The intercept helps position the line. Zero-item orders are outside our observed order range, so its numerical value should not automatically be interpreted as a real warehouse's fixed preparation time.

`LinearRegression` uses ordinary least squares: it chooses parameters that minimise the sum of squared errors on training rows. We still evaluate and select using MAE. The fitting objective and the evaluation criterion answer different questions and need not be identical.

In [ ]:
model_one = LinearRegression()
model_one.fit(X_train_one, y_train)

prediction_one = model_one.predict(X_validation_one)
mae_one = mean_absolute_error(y_validation, prediction_one)

print(f"Intercept: {model_one.intercept_:.2f} minutes")
print(f"Item coefficient: {model_one.coef_[0]:.2f} minutes per item")
print(f"Items-only validation MAE: {mae_one:.2f} minutes")

The attributes `intercept_` and `coef_` contain the fitted parameters. scikit-learn uses trailing underscores for many attributes learned during fitting. `coef_` is an array in feature-column order; `[0]` selects the first coefficient.

We can check how the fitted object uses those quantities on one input. This is a check of the prediction calculation, not a check of accuracy: we have no observed outcome for this illustrative new order.

In [ ]:
five_item_order = pd.DataFrame({"item_count": [5]})

manual_prediction = model_one.intercept_ + model_one.coef_[0] * 5
method_prediction = model_one.predict(five_item_order)[0]

print(f"From the line formula: {manual_prediction:.2f} minutes")
print(f"From predict(): {method_prediction:.2f} minutes")

**Self-check:** If the item coefficient is positive, what happens to the prediction when the item count increases by one? Does the coefficient prove the causal effect of adding an item?

The prediction increases by that coefficient. It does not establish a causal effect. Larger orders also tend to differ in weight and package size, so the fitted slope can reflect several related characteristics.

The line is an approximation to the relationship in the selected inputs. It is not the formula that generated the CSV, and its coefficients should not be expected to reproduce the generator's individual contributions.

<h2 id="section-add-a-second-feature">8. Add a second feature</h2>

The second regression uses both counts:

**predicted minutes = intercept + item coefficient × total items + fragile coefficient × fragile items**

Holding total items fixed while increasing fragile items means replacing a non-fragile item with a fragile one. Adding a new fragile item instead increases **both** counts. These are different input changes.

We keep the same training and validation rows and the same targets. The only modelling change is the input feature set. This makes the comparison fair: a different random allocation could otherwise explain part of the score difference.

In [ ]:
X_train_two = train[features_two]
X_validation_two = validation[features_two]

model_two = LinearRegression()
model_two.fit(X_train_two, y_train)

prediction_two = model_two.predict(X_validation_two)
mae_two = mean_absolute_error(y_validation, prediction_two)

coefficients = pd.DataFrame({
    "feature": features_two,
    "coefficient": model_two.coef_,
})
print(f"Intercept: {model_two.intercept_:.2f} minutes")
print(coefficients)
print(f"Two-feature validation MAE: {mae_two:.2f} minutes")

Both regressions still use a restricted view of a richer process. Knowing fragile item count may help, but it does not reveal the order's actual weight, package size, or wrapping requirements.

**Self-check:** Does adding an input guarantee a lower validation MAE?

No. Least-squares fitting can use an additional coefficient to improve or preserve training squared-error fit, but that does not guarantee lower MAE on unseen observations. An extra feature can contribute little useful information or fit sample-specific variation. We judge its contribution using validation results.

<h2 id="section-compare-predictions-and-errors">9. Compare predictions and errors</h2>

All three candidates are evaluated against the same validation outcomes. The first two numerical columns below are different views of the same comparison: MAE in minutes and reduction relative to the baseline.

Keep full precision for selecting a candidate. `.round(3)` changes the displayed table, not the original values stored in `comparison`.

In [ ]:
candidate_names = ["Median baseline", "Items only", "Items + fragile"]

comparison = pd.DataFrame({
    "candidate": candidate_names,
    "validation_MAE_minutes": [mae_baseline, mae_one, mae_two],
})
comparison["improvement_over_baseline_minutes"] = (
    mae_baseline - comparison["validation_MAE_minutes"]
)
comparison.round(3)

In [ ]:
# Quantify the gain from adding fragile count to the items-only model.
extra_gain_minutes = mae_one - mae_two
extra_gain_seconds = 60 * extra_gain_minutes
relative_gain = extra_gain_minutes / mae_one

print(f"Gain from the second feature: {extra_gain_minutes:.3f} minutes")
print(f"Equivalent average reduction: {extra_gain_seconds:.2f} seconds")
print(f"Relative MAE reduction: {relative_gain:.2%}")

For the supplied data and split, the baseline MAE is about 8.04 minutes. The one-feature regression reduces it to about 3.25 minutes; the two-feature regression gives about 3.22 minutes. The large improvement comes from using total item count. The second feature contributes only about 0.031 minutes—roughly 1.9 seconds or 1%—to the average error reduction.

This is a small **observed** improvement, not evidence that every individual order improves or that the same advantage will appear on another sample. Its practical value depends on how predictions are used and how reliably the extra information can be collected.

**Self-check:** Is “lowest validation MAE” the same statement as “clearly better for warehouse planning”?

No. The first follows our numerical selection rule. The second requires considering the improvement's size, operational costs, and uncertainty. We use the stated rule for the worked example while keeping that distinction visible.

### Learn to read one actual-versus-predicted plot

A prediction plot can evaluate a model with any number of inputs. Its axes show **actual duration** and **predicted duration**, not the input features.

Consider a point at (10, 8): the order actually took 10 minutes, but the model predicted 8. It lies below the diagonal `y = x`, so the model underestimated by 2 minutes. A point at (10, 12) overestimates by 2 minutes. A point on the diagonal is correct.

The vertical distance to the diagonal, measured in minutes, is the signed error. MAE averages the absolute vertical distances. The baseline produces a horizontal band because all predictions equal the same constant.

### Fix the scales for a fair visual comparison

We will use the same limits and colour mapping in every validation panel.

`np.concatenate` joins the actual and predicted arrays into one sequence so we can find shared limits. `plt.Normalize` maps fragile counts onto a common colour scale. Without a shared scale, the same colour could mean different counts in different panels.

The actual values and predictions refer to the same rows in the same order. We convert the actual Series to an array with `.to_numpy()` when doing position-by-position comparisons.

In [ ]:
actual_validation = y_validation.to_numpy()
validation_predictions = [
    prediction_baseline,
    prediction_one,
    prediction_two,
]

all_validation_times = np.concatenate(
    [actual_validation] + validation_predictions
)
plot_lower = min(0, all_validation_times.min() - 1)
plot_upper = all_validation_times.max() + 1

validation_colours = validation["fragile_count"].to_numpy()
colour_scale = plt.Normalize(
    vmin=0,
    vmax=max(1, validation_colours.max()),
)

### Reuse the same drawing instructions

The function below draws one panel on a supplied `Axes` object. Its arguments specify the observations, predictions, label, and common scales. It returns the scatter-plot object so that the surrounding figure can add a colour bar.

This applies Week 1's functions to Week 2's plotting objects. `set_aspect("equal")` gives one minute the same visual length on both axes. Together with equal limits, it makes the perfect-prediction diagonal easy to interpret.

In [ ]:
def draw_prediction_panel(ax, actual, predicted, fragile_counts,
                          title, lower, upper, colour_norm):
    # Plot one point per order, using a shared count-to-colour mapping.
    points = ax.scatter(
        actual,
        predicted,
        c=fragile_counts,
        cmap="viridis",
        norm=colour_norm,
        alpha=0.6,
        s=24,
    )

    # Two endpoints define the perfect-prediction reference line.
    ax.plot([lower, upper], [lower, upper],
            "--", color="grey", label="Perfect prediction")

    ax.set_xlim(lower, upper)
    ax.set_ylim(lower, upper)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Actual packing time (minutes)")
    ax.set_title(title)
    ax.grid(alpha=0.2)

    return points

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")

points = draw_prediction_panel(
    ax,
    actual_validation,
    prediction_one,
    validation_colours,
    f"Items only — validation MAE {mae_one:.2f} min",
    plot_lower,
    plot_upper,
    colour_scale,
)
ax.set_ylabel("Predicted packing time (minutes)")
ax.legend()
fig.colorbar(points, ax=ax, label="Fragile items")
plt.show()

**Self-check:** Is a point below the diagonal an overestimate or an underestimate? Does a point's colour tell you its error?

Below means an underestimate: predicted time is smaller than actual time. Colour tells you fragile count, not error. Error is represented by the vertical displacement from the diagonal.

We can now compare all candidates using the same visual grammar.

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5),
    sharex=True,
    sharey=True,
    layout="constrained",
)

# zip pairs each panel with its predictions and candidate name.
for ax, predicted, name in zip(
    axes, validation_predictions, candidate_names
):
    panel_mae = mean_absolute_error(y_validation, predicted)
    points = draw_prediction_panel(
        ax,
        actual_validation,
        predicted,
        validation_colours,
        f"{name}\nValidation MAE = {panel_mae:.2f} min",
        plot_lower,
        plot_upper,
        colour_scale,
    )

axes[0].set_ylabel("Predicted packing time (minutes)")
fig.colorbar(points, ax=axes, label="Fragile items", shrink=0.8)
plt.show()

### Interpret the comparison

The baseline cannot distinguish short and long orders, so its predictions form a horizontal band. The regression panels follow actual duration more closely, but still contain substantial deviations.

The items-only panel has horizontal bands because item counts are whole numbers: every order with the same count receives the same prediction, even when its actual duration differs. This is a consequence of the model's restricted inputs, not a plotting error.

The two regression panels may look very similar. That agrees with the small MAE difference; we should not exaggerate a visual advantage of the second feature. Colour may reveal groups of errors, but a clean colour pattern is not guaranteed.

A prediction plot also exposes individual large errors that a single average hides. Similar MAEs can conceal different distributions of mistakes, so the number and the plot provide complementary information.

### Where do errors occur?

A **signed-error plot** places prediction minus actual duration on the vertical axis. The zero line replaces the perfect-prediction diagonal.

Here the horizontal axis is **total items**, and colour still represents **fragile items**. This asks whether particular input regions show systematic over- or underprediction. We use the same vertical limits and colour scale for every candidate.

In [ ]:
validation_errors = []
for predicted in validation_predictions:
    validation_errors.append(predicted - actual_validation)

all_errors = np.concatenate(validation_errors)
error_limit = np.abs(all_errors).max() + 1

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 4.5),
    sharex=True,
    sharey=True,
    layout="constrained",
)

for ax, error, name in zip(axes, validation_errors, candidate_names):
    points = ax.scatter(
        validation["item_count"],
        error,
        c=validation_colours,
        cmap="viridis",
        norm=colour_scale,
        alpha=0.6,
        s=24,
    )
    ax.axhline(0, color="grey", linestyle="--")
    ax.set_ylim(-error_limit, error_limit)
    ax.set_xlabel("Total items")
    ax.set_title(name)
    ax.grid(alpha=0.2)

axes[0].set_ylabel("Predicted − actual time (minutes)")
fig.colorbar(points, ax=axes, label="Fragile items")
plt.show()

**Self-check:** If errors are centered on zero for large orders, does that mean predictions for those orders are accurate?

Not necessarily. Over- and underpredictions can balance while individual errors remain large. Look at the spread as well as the centre.

A constant baseline tends to overpredict small orders and underpredict large ones. Regression reduces that broad mismatch. Remaining spread can arise from omitted information, an inadequate functional form, and random variation. For example, equal item counts do not imply equal weights or package sizes.

This plot alone cannot separate those explanations. It can motivate a future investigation, but adding features or trying a new model would be a new selection step requiring suitable validation.

<h2 id="section-select-refit-and-test">10. Select, refit, and test</h2>

We now apply the rule stated before comparison: select the candidate with the smallest validation MAE. `.idxmin()` returns the row label of the smallest value; with this table's order it also implements our preference for the simpler candidate in an exact tie.

The selected approach includes both the model type and the feature list. Test outcomes play no role in this decision.

In [ ]:
selected_index = comparison["validation_MAE_minutes"].idxmin()
selected_name = comparison.loc[selected_index, "candidate"]
selected_validation_mae = comparison.loc[
    selected_index, "validation_MAE_minutes"
]

candidate_features = [features_one, features_one, features_two]
final_features = candidate_features[selected_index]

print("Selected candidate:", selected_name)
print("Selected features:", final_features)
print(f"Selection MAE: {selected_validation_mae:.3f} minutes")

The two-feature regression wins the specified rule for these data, but its advantage over the one-feature version is small. We record both facts rather than treating a numerical winner as a universal recommendation.

### Refit the selected approach

During comparison, each candidate learned from only the training subset. Now that the choice is fixed, we can learn the selected approach's parameters again from **training and validation together**.

This is called **refitting**. It uses more development examples for the final parameter estimates. It does not revisit the feature or model choice. The test set remains excluded.

We create a fresh object so that the final fit is explicit. The conditional supports whichever of our three candidates the selection rule chooses.

In [ ]:
if selected_index == 0:
    final_model = DummyRegressor(strategy="median")
else:
    final_model = LinearRegression()

final_model.fit(
    development[final_features],
    development[target],
)

print("Final fitting observations:", len(development))
print("Test observations still reserved:", len(test))

### Evaluate the fixed choice on held-out orders

Only now do we predict and measure errors on the test set. The result assesses the chosen and refitted approach, not every candidate.

If the result disappoints, report and investigate its limitations. Choosing a different candidate because of this test score would let the test influence selection. That revised procedure would need another independent final assessment.

Re-running the same fixed notebook to check reproducibility is different from repeatedly changing modelling decisions in response to test results.

In [ ]:
test_predictions = final_model.predict(test[final_features])
test_mae = mean_absolute_error(test[target], test_predictions)

print(f"Selected candidate: {selected_name}")
print(f"Validation MAE before refitting: {selected_validation_mae:.3f} minutes")
print(f"Final held-out test MAE: {test_mae:.3f} minutes")

The test MAE estimates average absolute error on unseen orders from this simulated process. It does not guarantee accuracy for every order or establish performance in a real warehouse.

Validation and test scores may differ because they concern different observations. The final model also used more fitting data. A difference in either direction is therefore possible; it is not by itself proof that something went wrong.

For the final figure we calculate limits from the test outcomes and their predictions **only for display**. These display choices do not change the model or reopen selection.

In [ ]:
actual_test = test[target].to_numpy()
test_plot_values = np.concatenate([actual_test, test_predictions])
test_lower = min(0, test_plot_values.min() - 1)
test_upper = test_plot_values.max() + 1
test_colours = test["fragile_count"].to_numpy()
test_colour_scale = plt.Normalize(
    vmin=0,
    vmax=max(1, test_colours.max()),
)

fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
points = draw_prediction_panel(
    ax,
    actual_test,
    test_predictions,
    test_colours,
    f"{selected_name}\nTest MAE = {test_mae:.2f} min",
    test_lower,
    test_upper,
    test_colour_scale,
)
ax.set_ylabel("Predicted packing time (minutes)")
fig.colorbar(points, ax=ax, label="Fragile items")
plt.show()

**Self-check:** Suppose another candidate would have a smaller test MAE. Should we now switch to it and still call this an independent test of the winner?

No. Using test results to choose the winner turns that set into selection data. Our procedure evaluates the choice already made using validation. A strong project reports the decision procedure and its limitations, not just the smallest score it can find.

<h2 id="section-predict-new-orders">11. Predict new orders</h2>

We can now supply inputs for orders whose actual durations are not yet known. Keep feature names and column order consistent with fitting. Selecting `new_orders[final_features]` handles both.

<table align="center" style="margin-left: auto; margin-right: auto;">
<thead>
<tr><th>Order</th><th style="text-align: right;">Total items</th><th style="text-align: right;">Fragile items</th></tr>
</thead>
<tbody>
<tr><td>A</td><td style="text-align: right;">5</td><td style="text-align: right;">0</td></tr>
<tr><td>B</td><td style="text-align: right;">5</td><td style="text-align: right;">3</td></tr>
<tr><td>C</td><td style="text-align: right;">15</td><td style="text-align: right;">3</td></tr>
</tbody>
</table>

**Self-check:** Predict the direction of the differences before running. A to B holds total items fixed and changes their composition. B to C adds ten non-fragile items. What does the fitted model expect?

In [ ]:
new_orders = pd.DataFrame({
    "item_count": [5, 5, 15],
    "fragile_count": [0, 3, 3],
}, index=["A", "B", "C"])

new_predictions = final_model.predict(new_orders[final_features])

prediction_table = new_orders.copy()
prediction_table["predicted_minutes"] = new_predictions
prediction_table.round(2)

For the selected two-feature model, B is predicted to take longer than A, and C longer than B. The exact differences follow its refitted coefficients. The values are model-based estimates for the inputs, not measured outcomes.

A baseline would give the same prediction to all three orders. An items-only model would give the same prediction to A and B. These differences show how the input choice limits what a model can distinguish.

**Self-check:** Can we calculate MAE for these new orders?

Not yet. MAE requires observed durations to compare with predictions. We would first need to pack the orders and measure how long they actually take. Calculating predictions alone does not evaluate their accuracy.

### Focused example: an order with 100 items

Now consider **100 total items, including 20 fragile items**. The inputs are logically consistent: fragile items do not exceed total items.

Run the model, then compare this order with the ranges in the data used for the final fit. `.agg(["min", "max"])` applies both summaries to each selected column.

In [ ]:
large_order = pd.DataFrame({
    "item_count": [100],
    "fragile_count": [20],
})

large_prediction = final_model.predict(large_order[final_features])[0]
print(f"Predicted duration: {large_prediction:.2f} minutes")
print("\nRanges in the final fitting data:")
print(development[features_two].agg(["min", "max"]))

**Self-check:**

1. Is 100 items within the range of the final fitting data?
2. Does returning a number make the prediction reliable?
3. What could change when packing such a large order?

The observed total-item range is 1–20. Extending a learned relationship far beyond observed inputs is **extrapolation**. The model continues its fitted rule, but we have not evaluated whether that rule is appropriate for such orders.

Large orders may need additional boxes, staff, space, or handling steps. The omitted package and weight information may become particularly consequential. A good test MAE on ordinary orders does not validate a prediction for 100 items. Decimal places express numerical formatting, not confidence.

This is a specific limitation of applying the model to a different input region. It is not repaired simply by switching to a more complex model.

<h2 id="section-recap-and-outlook">12. Recap and outlook</h2>

Our workflow was:

```text
define the question and prediction time
→ select meaningful features and target
→ check the table
→ reserve training, validation, and test rows
→ fit a baseline and two regressions
→ compare validation MAE and error patterns
→ fix the choice and refit on development data
→ evaluate on test data
→ predict new orders and examine an extrapolation example
```

Week 4 uses the **same CSV with all seven candidate input features**, but also changes the prediction question.

<h2 id="section-optional-your-turn">Optional: Your turn</h2>

The complete walkthrough ends above. These investigations let you make and justify your own choices. Work with copies or fresh model objects so that the walkthrough results remain reproducible.

The test set has already been inspected. Treat further model changes as exploratory; a newly selected approach would need an independent final assessment. Use training and validation for these investigations.

1. **Practical relevance.** Imagine recording fragile count costs staff time. Use the validation results to argue whether the extra feature is worthwhile. What additional evidence or business information would strengthen your argument? Distinguish the arithmetic winner from a practical recommendation.
2. **Individual errors.** Create a validation table containing both feature counts, actual duration, predicted duration, and absolute error for each regression. Find orders where the second model helps and orders where it hurts. Explain why a lower MAE does not imply an improvement on every order.
3. **Sensitivity to the split.** Keep the original `test` data aside. Split only `development` into training and validation using several other fixed seeds, refit all three candidates, and compare their validation MAEs. Report variation and any changes of ranking. Do not choose the seed that gives your preferred result. This motivates Week 4's cross-validation.
4. **MAE versus RMSE.** Compare error arrays `[2, 2]` and `[0, 4]` using MAE and root mean squared error. Use `np.sqrt(np.mean(errors ** 2))` for RMSE. Both measures have units of minutes, but RMSE penalises the concentrated large error more. Explain a decision context for each metric without changing the primary metric merely to favour a candidate.
5. **New order scenarios.** Predict three new plausible orders. Change one feature at a time and explain the differences using the fitted coefficients. Separate a numerical model estimate from evidence about the real outcome.

In [ ]:
# Optional exploration workspace
# Start with copies of tables or fresh model objects.

### Before you finish

1. Restart the kernel and run all cells from top to bottom.
2. Revisit any self-check whose answer was surprising.
3. Save the notebook. If you are working in Binder, download your notebook before closing the session.